## Save 3 frames images per second into raw_frames folder for manual annotation in Roboflow

In [ ]:
import cv2
import os

# Import video
video = cv2.VideoCapture("video/sample.mp4")   

# Create raw_frames folder if it does not exist 
try: 
    if not os.path.exists('raw_frames'):
        os.mkdir('raw_frames')
except OSError:
    print('Error: Creating directory of raw_frames')

fps = video.get(cv2.CAP_PROP_FPS) # Retrieve video's frames per second

interval = int(fps/3) 

frame_count = 0 
interval_count = 0 

# Save an image from the video for every interval 
while video.isOpened(): 
    success, frame = video.read()
    if not success: 
        break 
    if frame_count%interval == 0:
        name = './raw_frames/' + str(interval_count) + '.jpg'
        print('Creating...' + name)
        cv2.imwrite(name, frame)
        frame_count+=1
        interval_count+=1
    frame_count+=1

video.release()
cv2.destroyAllWindows()

## Save annotated images that split into train, validate, test data by calling Roboflow API

In [1]:
# Import manually labeled images of piano tag from Roboflow
import os
from roboflow import Roboflow

api_key = os.getenv("ROBOFLOW_API_KEY")
if not api_key:
    raise ValueError("Set ROBOFLOW_API_KEY in your environment before downloading from Roboflow.")

rf = Roboflow(api_key=api_key)
project = rf.workspace("genevieves-workspace-bdofm").project("staff-tag-detection-fmgbi")
version = project.version(1)
dataset = version.download("yolov11")
                

loading Roboflow workspace...
loading Roboflow project...


## Train model

In [ ]:
# Fine tune yolov8n with piano tag images 
from ultralytics import YOLO # Initialize the model with pre-trained weights

model = YOLO('yolov8n.pt') 

results = model.train(
    data='staff-tag-detection-1/data.yaml', 
    epochs=50, # Run the training process 50 times through the dataset
    imgsz=640, 
    device='cpu'
)

# Implementaion

In [ ]:
import cv2 
from ultralytics import YOLO
import pandas as pd 

# Import model for tag and person detections
tag_model = YOLO('runs/detect/train/weights/best.pt')
person_model = YOLO('best.pt')
person_tracker = "bytetrack_staff.yaml"
# person_model = YOLO('yolov8n.pt')

video = cv2.VideoCapture("video/sample.mp4")
out = cv2.VideoWriter(
    'output/output.mp4'
    , cv2.VideoWriter_fourcc(*'mp4v')
    , video.get(cv2.CAP_PROP_FPS)
    , (int(video.get(cv2.CAP_PROP_FRAME_WIDTH)), int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))))

if not video.isOpened():
    print("Error: Could not open video file.")
    exit()

staff_set = set()
staff_dict = {}
staff_frame_count = {}
staff_min_frames = 5 
frame_count = 0
records = []

while True: 
    ret, frame = video.read()
    if not ret: 
        break 

    frame_count += 1

    person_results = person_model.track(frame, persist = True, tracker = person_tracker, conf = 0.2) # Load model to detect and track person
    tag_results = tag_model.track(frame, persist=True) # Load model to detect and track piano tag

    # If piano tag's centre is inside a person bounding box for more than 5 frames, add into staff_set
    for t in tag_results[0].boxes: 
            tx1, ty1, tx2, ty2 = map(int, t.xyxy[0])
            tag_centreX = (tx2 + tx1)/2 
            tag_centreY = (ty2 + ty1)/2 
            confT = t.conf[0]

            if confT>0.2: 
                cv2.rectangle(frame, (tx1, ty1), (tx2, ty2), (0, 255, 0), 2)
                cv2.putText(frame, f"Tag {confT:.2f}", (tx1, max(ty1 - 8, 0)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                
                for p in person_results[0].boxes: 
                    if p.id is None:
                        continue 
                    px1, py1, px2, py2 = map(int, p.xyxy[0])
                    confP = p.conf[0]
                    id = int(p.id[0])

                    if (confP >= 0.2) and (px1 < tag_centreX < px2 and py1 < tag_centreY < py2): # If person detection confidence is greater than or equal to 0.2, and tag is inside person detection bounding box
                        if (id not in staff_frame_count):
                            staff_frame_count[id] = 0 # To keep track if tag inside a person bounding box is greater than staff_min_frames
                        staff_frame_count[id] += 1

                        # If paino tag is inside a person for more than 5 frames and not recorded inside staff_sets, add the person ID into staff_set
                        if staff_frame_count[id] > staff_min_frames and id not in staff_set:
                            staff_set.add(id)
                        
                break

    total_seconds = frame_count / video.get(cv2.CAP_PROP_FPS)
    minutes = int(total_seconds // 60)
    seconds = int(total_seconds % 60)

    # For each person ID in staff_set, draw bounding box, calculate and store XY coordinate
    for p in person_results[0].boxes: 
        if p.id is None:
            continue 
        if (int(p.id[0]) in staff_set):
            px1, py1, px2, py2 = map(int, p.xyxy[0])
            confP = p.conf[0]

            cv2.rectangle(frame, (px1,py1),(px2,py2),(0,255,0),2)
            cv2.putText(frame, f"Staff {confP:.2f}",(px1, max(py1-8, 0)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

            # Calculation to get person's XY coordinate
            staff_x = int((px1+px2)/2)
            staff_y = int((py1+py2)/2)

            staff_info = {
                # "ID": id,
                "timestamp": f"{minutes:02d}:{seconds:02d}",
                "x": staff_x,
                "y": staff_y,
                "person_conf": round(float(confP), 2)
            }

            staff_dict[id] = staff_info
            records.append(staff_info)

    if staff_dict:
        # cv2.rectangle(frame, (10, 5), (230, 55), (255,255,255), -1)
        # cv2.putText(frame, f"ID: {list(staff_dict.keys())[-1]}", (15, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)
        # cv2.putText(frame, f"Coordinate(x,y): {staff_dict[list(staff_dict.keys())[-1]]['x']},{staff_dict[list(staff_dict.keys())[-1]]['y']}", (15, 45), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)
        
        cv2.rectangle(frame, (10, 5), (230, 35), (255,255,255), -1)
        cv2.putText(frame, f"Coordinate(x,y): {staff_dict[list(staff_dict.keys())[-1]]['x']},{staff_dict[list(staff_dict.keys())[-1]]['y']}", (15, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)

    out.write(frame)
    cv2.imshow("Staff Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

df = pd.DataFrame(records)
df.to_csv("output/staff.csv", index=False)
video.release()
out.release()
cv2.destroyAllWindows()


0: 480x640 18 peoples, 69.7ms
Speed: 2.6ms preprocess, 69.7ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 56.0ms
Speed: 1.6ms preprocess, 56.0ms inference, 0.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 17 peoples, 69.5ms
Speed: 2.0ms preprocess, 69.5ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 76.2ms
Speed: 1.5ms preprocess, 76.2ms inference, 0.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 16 peoples, 94.9ms
Speed: 1.6ms preprocess, 94.9ms inference, 1.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 266.3ms
Speed: 4.1ms preprocess, 266.3ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 17 peoples, 52.1ms
Speed: 2.3ms preprocess, 52.1ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 51.6ms
Speed: 1.4ms preprocess, 51.6ms inference, 0.2ms po